In [48]:
import pandas as pd 
import yaml

users_df = pd.read_csv("../data3/users.csv") 

with open("../data3/books.yaml") as books:
    data=yaml.safe_load(books) 
books_df = pd.DataFrame(data)

orders_df = pd.read_parquet("../data3/orders.parquet")


In [16]:
users_df["phone"] = users_df["phone"].str.replace(r"\D", "", regex=True)

In [17]:
users_df.head()

,id,name,address,phone,email
0,47856,Tammie Mayer,"Apt. 371 1875 Gusikowski Stravenue, Moenland, ...",1632122656,maire.larson@runolfsdottir-mclaughlin.example
1,49394,Evia Yost,"22610 Mildred Green, Pagacbury, AZ 19393-2103",7789266707,byron@johnston.test
2,48386,Margery Gorczany,"Apt. 884 349 Renner Lane, East Darren, NC 30837",9923755021,cletus@baumbach.test
3,49547,Edris Kshlerin,"Suite 498 24711 Bechtelar Extensions, Ryanvill...",7475372209,leonore@gleason.test
4,50512,Meghan Thiel,"125 Rodger Drive, Rogahnborough, VT 96988-4596",6317657676,cristine@bayer.test


In [18]:
users_df["phone_length"] = users_df["phone"].str.len()
users_df["phone_length"].value_counts()

phone_length
10    3466
Name: count, dtype: int64

In [19]:
users_df = users_df.drop(columns=['phone_length'])

In [20]:
import numpy as np
users_df['address'] = users_df['address'].replace(r'^\s*$', np.nan, regex=True)

In [21]:
books_df.columns = books_df.columns.str.strip(':')
books_df.columns

Index(['id', 'title', 'author', 'genre', 'publisher', 'year'], dtype='object')

In [22]:
books_df['publisher'] = books_df['publisher'].replace(r'^\s*$', np.nan, regex=True)
books_df['year'] = books_df['year'].replace(r'^\s*$', np.nan, regex=True)

In [23]:
books_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 762 entries, 0 to 761
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         762 non-null    int64 
 1   title      762 non-null    object
 2   author     762 non-null    object
 3   genre      762 non-null    object
 4   publisher  748 non-null    object
 5   year       756 non-null    object
dtypes: int64(1), object(5)
memory usage: 35.8+ KB


In [24]:
books_df['year'] = pd.to_numeric(books_df['year'], errors='coerce').astype('Int64')

In [25]:
import re

patterns = [
    r'\d{4}-\d{2}-\d{2}',                   
    r'\d{2}/\d{2}/\d{2,4}',              
    r'\d{1,2}-[A-Za-z]+-\d{4}',              
    r'\d{1,2}-[A-Za-z]{3}-\d{4}',           
    r'\d{1,2}\.\d{1,2}\.\d{4}',             
    r'[A-Za-z]{3}\s[A-Za-z]{3}\s+\d{1,2}\s[\d:]+\s\d{4}',  
]

def extract_date(ts):
    if pd.isnull(ts):
        return None
    ts = str(ts)
    for pattern in patterns:
        match = re.search(pattern, ts)
        if match:
            return match.group()
    return None

orders_df['date'] = orders_df['timestamp'].apply(extract_date)
orders_df['date'] = pd.to_datetime(orders_df['date'], errors='coerce')
print(orders_df['date'].isnull().sum())

5173


In [26]:
mask = orders_df['date'].isnull()
orders_df[mask]['timestamp'].head(30)

2         11:17:51 PM, 29-JUN-2025
3                   09:29;11/06/24
6                 6-JUL-2024,15:27
9                04:51:52,08/18/24
13         22-JAN-2025 09:29:50 pm
18                  16:03 12/04/24
19       16:05:16, 6-November-2024
20              11:45:28, 03/06/25
21          04/16/25;08:36:11 A.M.
23            08:57:54;10-Sep-2024
24                  13:50 12/07/24
25      03:29:22 P.M., 10-Apr-2025
26               02/05/25,16:14:04
27               10:30:57 08/24/24
31         06:02:55 AM,18-AUG-2024
33               13:06,10-JUL-2024
35       05:32:32 P.M.,14-Jan-2025
36              19:44:52, 12/23/24
37        3-June-2025, 11:53:37 AM
38                  08/10/24 23:23
41        Thu Mar 20 05:57:41 2025
42                  13:00 01/05/25
43              18-Nov-2024, 00:04
46               12/28/24,05:18:09
49            11:45:59 pm 04/09/24
53             10.02.2025 11:08:08
55              5.01.2025  1:19:19
56              01/25/25, 05:42:32
57    01:14:38 am; 2

In [27]:
import re
patterns = [
    r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+',     
    r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}',           
    r'\d{4}-\d{2}-\d{2}',                           
    r'\d{1,2}-[A-Z]{3}-\d{4}',                    
    r'\d{1,2}-[A-Za-z]+-\d{4}',                     
    r'\d{1,2}-[A-Za-z]{3}-\d{4}',                    
    r'\d{1,2}\.\d{1,2}\.\d{4}',                       
    r'\d{2}/\d{2}/\d{2,4}',                           
    r'[A-Za-z]{3}\s[A-Za-z]{3}\s+\d{1,2}\s[\d:]+\s\d{4}',  
]
def extract_date(ts):
    if pd.isnull(ts):
        return None
    ts = str(ts).strip()

    for pattern in patterns:
        match = re.search(pattern, ts)
        if match:
            return match.group()
    return None

orders_df['date'] = orders_df['timestamp'].apply(extract_date)
orders_df['date'] = pd.to_datetime(orders_df['date'], format='mixed', errors='coerce')
print(orders_df['date'].isnull().sum())

0


In [28]:
orders_df['date'] = pd.to_datetime(orders_df['date'], format='mixed', dayfirst=True, errors='coerce')
print(orders_df['date'].isnull().sum())

0


In [29]:
orders_df.head(10)

,id,user_id,book_id,quantity,unit_price,timestamp,shipping,date
0,80941,47864,21550,1,67.0 €,2025-01-11 09:50:11 PM,,2025-01-11 00:00:00.000
1,81245,47864,22052,2,EUR26.99,2024-03-27T07:46:55.055,,2024-03-27 07:46:55.055
2,78477,48918,21367,1,30 $,"11:17:51 PM, 29-JUN-2025",,2025-06-29 00:00:00.000
3,86303,48092,21724,1,18 $,09:29;11/06/24,None,2024-11-06 00:00:00.000
4,81990,48418,21441,1,USD 24.99,2024-09-16;03:05:39 pm,None,2024-09-16 00:00:00.000
5,82785,50265,21459,2,41.50$,2024-06-29T00:13:48.016,,2024-06-29 00:13:48.016
6,84594,49429,21598,1,19.99 $,"6-JUL-2024,15:27",NULL,2024-07-06 00:00:00.000
7,86155,49965,21752,1,61.99 $,"11:28:42 AM, 2025-01-08",,2025-01-08 00:00:00.000
8,82478,50188,21587,1,69.5$,2024-11-08T02:05:08.703,None,2024-11-08 02:05:08.703
9,80811,50433,21899,1,66.99 €,"04:51:52,08/18/24",None,2024-08-18 00:00:00.000


In [30]:
import re

def extract_price(val):
    if pd.isnull(val):
        return None, None
    val = str(val)
    if '$' in val or 'USD' in val:
        currency = 'USD'
    elif '€' in val or 'EUR' in val:
        currency = 'EUR'
    else:
        currency = 'UNKNOWN'
    cents_match = re.search(r'(\d+)\$(\d+)¢', val)
    if cents_match:
        amount = float(cents_match.group(1)) + float(cents_match.group(2)) / 100
        return amount, 'USD'
    match = re.search(r'[\d]+\.?\d*', val)
    if match:
        return float(match.group()), currency
    return None, currency

orders_df[['price_clean', 'currency']] = orders_df['unit_price'].apply(
    lambda x: pd.Series(extract_price(x))
)

print(orders_df[['unit_price', 'price_clean', 'currency']].head(5))
print(orders_df['currency'].value_counts())

  unit_price  price_clean currency
0     67.0 €        67.00      EUR
1   EUR26.99        26.99      EUR
2       30 $        30.00      USD
3       18 $        18.00      USD
4  USD 24.99        24.99      USD
currency
USD    6062
EUR    2871
Name: count, dtype: int64


In [31]:
mask = orders_df['price_clean'].isnull() | (orders_df['currency'] == 'UNKNOWN')
orders_df[mask]['unit_price'].head(20)

Series([], Name: unit_price, dtype: object)

In [32]:
rate = 1.2

orders_df['price_usd'] = orders_df.apply(
    lambda row: round(row['price_clean'] * rate, 2)
    if row['currency'] == 'EUR'
    else round(row['price_clean'], 2),
    axis=1
)

In [33]:
orders_df['paid_price'] = orders_df['quantity'] * orders_df['price_usd']
orders_df[['quantity','unit_price', 'price_clean', 'currency', 'price_usd','paid_price']].head(10)

,quantity,unit_price,price_clean,currency,price_usd,paid_price
0,1,67.0 €,67.00,EUR,80.40,80.40
1,2,EUR26.99,26.99,EUR,32.39,64.78
2,1,30 $,30.00,USD,30.00,30.00
3,1,18 $,18.00,USD,18.00,18.00
4,1,USD 24.99,24.99,USD,24.99,24.99
5,2,41.50$,41.50,USD,41.50,83.00
6,1,19.99 $,19.99,USD,19.99,19.99
7,1,61.99 $,61.99,USD,61.99,61.99
8,1,69.5$,69.50,USD,69.50,69.50
9,1,66.99 €,66.99,EUR,80.39,80.39


In [34]:
orders_df['shipping'] = orders_df['shipping'].replace(r'^\s*$', np.nan, regex=True)
orders_df = orders_df.replace({None: np.nan})
orders_df['shipping'].isna().sum()

np.int64(4473)

In [35]:
orders_df = orders_df.drop(columns=['unit_price','timestamp'])

In [36]:
orders_df['shipping'] = orders_df['shipping'].replace("NULL", np.nan)
orders_df['shipping'].eq("NULL").sum()

np.int64(0)

In [38]:
orders_df.head(10)

,id,user_id,book_id,quantity,shipping,date,price_clean,currency,price_usd,paid_price
0,80941,47864,21550,1,NaN,2025-01-11 00:00:00.000,67.00,EUR,80.40,80.40
1,81245,47864,22052,2,NaN,2024-03-27 07:46:55.055,26.99,EUR,32.39,64.78
2,78477,48918,21367,1,NaN,2025-06-29 00:00:00.000,30.00,USD,30.00,30.00
3,86303,48092,21724,1,NaN,2024-11-06 00:00:00.000,18.00,USD,18.00,18.00
4,81990,48418,21441,1,NaN,2024-09-16 00:00:00.000,24.99,USD,24.99,24.99
5,82785,50265,21459,2,NaN,2024-06-29 00:13:48.016,41.50,USD,41.50,83.00
6,84594,49429,21598,1,NaN,2024-07-06 00:00:00.000,19.99,USD,19.99,19.99
7,86155,49965,21752,1,NaN,2025-01-08 00:00:00.000,61.99,USD,61.99,61.99
8,82478,50188,21587,1,NaN,2024-11-08 02:05:08.703,69.50,USD,69.50,69.50
9,80811,50433,21899,1,NaN,2024-08-18 00:00:00.000,66.99,EUR,80.39,80.39


In [39]:
orders_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8933 entries, 0 to 8932
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   id           8933 non-null   int64         
 1   user_id      8933 non-null   int64         
 2   book_id      8933 non-null   int64         
 3   quantity     8933 non-null   int32         
 4   shipping     2272 non-null   object        
 5   date         8933 non-null   datetime64[ns]
 6   price_clean  8933 non-null   float64       
 7   currency     8933 non-null   object        
 8   price_usd    8933 non-null   float64       
 9   paid_price   8933 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int32(1), int64(3), object(2)
memory usage: 663.1+ KB


In [40]:
(users_df == 'NULL').any()

id         False
name       False
address    False
phone      False
email      False
dtype: bool

In [41]:
(books_df=='NULL').any()

id           False
title        False
author       False
genre        False
publisher     True
year         False
dtype: boolean

In [42]:
books_df['publisher'] = books_df['publisher'].replace("NULL", np.nan)

In [43]:
(books_df=='NULL').any()

id           False
title        False
author       False
genre        False
publisher    False
year         False
dtype: boolean

In [44]:
(orders_df=='NULL').any()

id             False
user_id        False
book_id        False
quantity       False
shipping       False
date           False
price_clean    False
currency       False
price_usd      False
paid_price     False
dtype: bool

In [45]:
books_df.head()

,id,title,author,genre,publisher,year
0,21778,Call of Duty: World at War,Elmer Parker,Mythopoeia,Orion Books,2003
1,21374,WarioWare: Touched!,Ms. Kami Prosacco,Mythology,St. Martin's Press,1951
2,21457,Titanic: Music from the Motion Picture,Zoila Christiansen,Historical fiction,"Farrar, Straus & Giroux",2009
3,21881,Flashdance: Original Soundtrack from the Motio...,"Joseph Raynor, Ja Ankunding II",Metafiction,Pen and Sword Books,2024
4,21446,Riot!,Louanne Cruickshank,Mystery,Bellevue Literary Press,1987


In [46]:
users_df.head()

,id,name,address,phone,email
0,47856,Tammie Mayer,"Apt. 371 1875 Gusikowski Stravenue, Moenland, ...",1632122656,maire.larson@runolfsdottir-mclaughlin.example
1,49394,Evia Yost,"22610 Mildred Green, Pagacbury, AZ 19393-2103",7789266707,byron@johnston.test
2,48386,Margery Gorczany,"Apt. 884 349 Renner Lane, East Darren, NC 30837",9923755021,cletus@baumbach.test
3,49547,Edris Kshlerin,"Suite 498 24711 Bechtelar Extensions, Ryanvill...",7475372209,leonore@gleason.test
4,50512,Meghan Thiel,"125 Rodger Drive, Rogahnborough, VT 96988-4596",6317657676,cristine@bayer.test


In [47]:
books_df.to_csv('../data3/books_clean.csv', index=False)
users_df.to_csv('../data3/users_clean.csv', index=False)
orders_df.to_csv('../data3/orders_clean.csv', index=False)